In [8]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.data_loading import TARGET_COL, ID_COL, PROCESSED_DATA_PATH, load_data_with_secondary_tables
from src.evaluation import get_feature_importance
from src.tuning import (
    tune_lgbm_fast,
    confirm_lgbm_candidates_cv,
    save_lgbm_params,
)

application_train, application_test = load_data_with_secondary_tables()
X = application_train[[c for c in application_train.columns if c not in [TARGET_COL, ID_COL]]]
y = application_train[TARGET_COL]

X.shape

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


(307511, 568)

In [9]:
N_TRIALS = 80
TOP_N = 8
TUNING_SAMPLE_SIZE = 180_000
CONFIRM_N_SPLIT = 5
EARLY_STOPPING_ROUNDS = 80
RANDOM_STATE = 42

In [10]:
tuning_results = tune_lgbm_fast(
    X,
    y,
    n_trials=N_TRIALS,
    top_n=TOP_N,
    sample_size=TUNING_SAMPLE_SIZE,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    random_state=RANDOM_STATE,
)

tuning_results['top_trials']

,candidate,trial_number,fast_auc,best_iteration
0,1,41,0.785090,1984
1,2,72,0.785066,2000
2,3,36,0.785021,1988
3,4,63,0.784852,1995
4,5,46,0.784838,1687
5,6,18,0.784810,919
6,7,31,0.784762,1806
7,8,33,0.784706,1893


In [11]:
candidate_cv_table, candidate_cv_results = confirm_lgbm_candidates_cv(
    X,
    y,
    tuning_results['top_params'],
    n_split=CONFIRM_N_SPLIT,
    random_state=RANDOM_STATE,
)

candidate_summary = tuning_results['top_trials'].merge(
    candidate_cv_table,
    on='candidate',
    how='left',
).sort_values('oof_auc', ascending=False)

candidate_summary

,candidate,trial_number,fast_auc,best_iteration,mean_auc,std_auc,oof_auc
1,2,72,0.785066,2000,0.790989,0.003603,0.790964
7,8,33,0.784706,1893,0.790884,0.003670,0.790856
2,3,36,0.785021,1988,0.790821,0.003463,0.790799
5,6,18,0.784810,919,0.790776,0.003434,0.790756
3,4,63,0.784852,1995,0.790568,0.003582,0.790549
6,7,31,0.784762,1806,0.790549,0.003366,0.790526
0,1,41,0.785090,1984,0.790539,0.003346,0.790521
4,5,46,0.784838,1687,0.790153,0.003605,0.790131


In [12]:
best_candidate = int(candidate_summary.iloc[0]['candidate'])
best_lgbm_params = tuning_results['top_params'][best_candidate - 1]
best_results = candidate_cv_results[best_candidate - 1]

best_params_path = save_lgbm_params(best_lgbm_params, 'best_lgbm_params.json')
top_params_path = save_lgbm_params(tuning_results['top_params'], 'top_lgbm_params.json')
candidate_summary_path = PROCESSED_DATA_PATH / 'lgbm_tuning_candidate_summary.csv'
oof_path = PROCESSED_DATA_PATH / 'lgbm_engineered_oof.csv'

candidate_summary.to_csv(candidate_summary_path, index=False)
pd.DataFrame({
    ID_COL: application_train[ID_COL],
    TARGET_COL: y,
    'OOF_PRED': best_results['oof_preds'],
}).to_csv(oof_path, index=False)
{
    'best_params_path': best_params_path,
    'top_params_path': top_params_path,
    'candidate_summary_path': candidate_summary_path,
    'oof_path': oof_path,
    'best_lgbm_params': best_lgbm_params,
    'fold_aucs': best_results['fold_aucs'],
}

{'best_params_path': PosixPath('/Users/ryanyao/Quant Projects/Home_Credit/data/processed/best_lgbm_params.json'),
 'top_params_path': PosixPath('/Users/ryanyao/Quant Projects/Home_Credit/data/processed/top_lgbm_params.json'),
 'candidate_summary_path': PosixPath('/Users/ryanyao/Quant Projects/Home_Credit/data/processed/lgbm_tuning_candidate_summary.csv'),
 'oof_path': PosixPath('/Users/ryanyao/Quant Projects/Home_Credit/data/processed/lgbm_engineered_oof.csv'),
 'best_lgbm_params': {'num_leaves': 30,
  'learning_rate': 0.009105520009865755,
  'min_child_samples': 131,
  'subsample': 0.8186257413768864,
  'colsample_bytree': 0.7653499767023243,
  'reg_alpha': 0.0014749451671424974,
  'reg_lambda': 2.114355223494756,
  'max_depth': 10,
  'class_weight': None,
  'subsample_freq': 1,
  'n_estimators': 2000},
 'fold_aucs': [0.7882208321101009,
  0.7965148330084119,
  0.7884508502403786,
  0.7940601455532484,
  0.7876963923900435]}

In [13]:
feature_importance = get_feature_importance(best_results['models'])
feature_importance_path = PROCESSED_DATA_PATH / 'lgbm_engineered_feature_importance.csv'
feature_importance.to_csv(feature_importance_path, index=False)
feature_importance.head(30)

,index,feature,fold_1,fold_2,fold_3,fold_4,fold_5,mean_importance,std_importance
0,305,EXT_SOURCE_MEAN,377879.110228,389582.697759,396023.752150,372321.627142,405808.670024,388323.171461,13525.493014
1,429,ORGANIZATION_TYPE,111356.876149,113660.402980,116339.452696,108484.099613,113394.618376,112647.089963,2924.648906
2,306,EXT_SOURCE_MIN,63892.899485,58815.918362,66402.230798,63362.454935,66660.313947,63826.763505,3161.904701
3,294,EXT_SOURCE_2_3_PROD,40780.676394,42190.300076,38095.931726,42804.603709,35019.011817,39778.104744,3219.557569
4,304,EXT_SOURCE_MAX,37181.409416,30889.197535,31218.435503,35682.575358,34573.193258,33908.962214,2768.290531
5,396,INSTALLMENTS_PAYMENT_LATE_MEAN,22115.956656,27708.698829,23408.775502,25941.762258,24331.309762,24701.300601,2183.730002
6,133,CREDIT_ANNUITY_RATIO,25267.993093,22558.175383,24570.812156,26377.997874,19603.348516,23675.665404,2667.955432
7,346,GOODS_CREDIT_RATIO,23042.626073,24205.287422,22988.212981,22139.353953,19036.339267,22282.363939,1957.740032
8,121,BUREAU_DEBT_CREDIT_MEAN_RATIO,21176.552458,20186.610481,21096.861134,22881.016087,20766.303586,21221.468749,1006.199391
9,428,OCCUPATION_TYPE,20789.060069,18998.465131,19208.853683,19969.332687,19055.460110,19604.234336,768.365825
